# Baseline JIT Defect Prediction Experiment

This notebook demonstrates a basic Just-In-Time (JIT) defect prediction experiment using the KG-Commit framework.

## Setup and Imports

In [ ]:
# Load configuration
config = Config.from_yaml("../../experiments/configs/default.yaml")
logger = SimpleLogger()

# Extract config values
data_path = config.get("data.test_path")
label_column = config.get("preprocessing.label_column")
timestamp_column = config.get("data.timestamp_column")
window_size = config.get("streaming.window_size")
window_step = config.get("streaming.window_step")
output_dir = Path(config.get("output.results_dir", "outputs/results"))
output_dir.mkdir(parents=True, exist_ok=True)

print("Configuration loaded:")
print(f"  Data path: {data_path}")
print(f"  Label column: {label_column}")
print(f"  Window size: {window_size}")
print(f"  Output directory: {output_dir}")

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..', '..'))

from kg_commit.data.dataset import CSVCommitDataset
from kg_commit.data.preprocess import SimplePreprocessor
from kg_commit.data.stream import CommitStream
from kg_commit.evaluation.evaluator import Evaluator
from kg_commit.model.online_model import OnlineModel
from kg_commit.training.incremental_trainer import IncrementalTrainer
from kg_commit.utils.config import Config
from kg_commit.persistence.serializer import JSONSerializer
from kg_commit.utils.logging import SimpleLogger

import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

## Data Loading

Load the dataset and examine its structure.

In [ ]:
# Load the dataset using config
dataset = CSVCommitDataset(source=data_path, label_column=label_column, timestamp_column=timestamp_column)
commits = dataset.load()

print(f"Loaded {len(commits)} commits")
print(f"Metadata: {dataset.get_metadata()}")

# Display first few commits
for i, commit in enumerate(commits[:3]):
    print(f"Commit {i+1}: {commit}")

## Preprocessing

Set up the preprocessor and stream the data into windows.

In [ ]:
# Set up preprocessing from config
preprocessor = SimplePreprocessor(
    label_column=label_column, 
    include_text=config.get("preprocessing.include_text", True)
)

# Create data stream from config
stream = CommitStream(commits, window_size=window_size, step=window_step)

print(f"Preprocessing configured with window_size={window_size}, step={window_step}")
print("Processing first window...")
first_window = next(iter(stream))
print(f"First window: {first_window}")

# Preprocess the window
preprocessed_window = preprocessor.transform_window(first_window)
print(f"Preprocessed window shape: {preprocessed_window.get_features().shape if preprocessed_window.get_features() is not None else 'None'}")
print(f"Labels shape: {preprocessed_window.get_labels().shape}")

## Model Training and Evaluation

Set up the model, evaluator, and trainer, then run the experiment.

In [ ]:
# Set up model and evaluator from config
model_classes = config.get("model.classes", [0, 1])
model = OnlineModel(classes=model_classes)
evaluator = Evaluator()

# Create trainer
trainer = IncrementalTrainer(model=model, preprocessor=preprocessor, evaluator=evaluator)

# Reset stream for full experiment
stream = CommitStream(commits, window_size=window_size, step=window_step)

# Run the experiment
print(f"Running experiment with {len(commits)} commits...")
results = []
for index, (window, preds) in enumerate(trainer.run(stream), start=1):
    if index % max(1, len(list(stream))//10) == 0 or index == 1:
        logger.log_window(window)
    results.append((window, preds))

# Get final evaluation
summary = evaluator.summarize()
print("\n" + "="*50)
print("Final Results:")
print("="*50)
print(f"Accuracy: {summary['accuracy']:.4f}")
print(f"Precision: {summary['precision']:.4f}")
print(f"Recall: {summary['recall']:.4f}")
print(f"F1 Score: {summary['f1']:.4f}")
print("\nClassification Report:")
print(summary['report'])

## Saving Results

In [ ]:
# Save results using config
serializer = JSONSerializer()
results_data = {
    "experiment_name": config.get("experiment.name"),
    "description": config.get("experiment.description"),
    "data_path": str(data_path),
    "window_size": window_size,
    "window_step": window_step,
    "num_windows": len(results),
    "summary": summary
}
results_file = output_dir / "baseline_results.json"
serializer.save(results_data, str(results_file))
print(f"Results saved to {results_file}")

## Visualization

Plot some results.

In [ ]:
# Simple visualization of predictions vs actual
window_indices = list(range(1, len(results) + 1))
accuracies = []

for window, preds in results:
    y_true = window.get_labels()
    acc = (preds == y_true).mean()
    accuracies.append(acc)

plt.figure(figsize=(10, 6))
plt.plot(window_indices, accuracies, marker='o')
plt.title('Accuracy per Window')
plt.xlabel('Window Index')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

print(f"Average accuracy across windows: {sum(accuracies)/len(accuracies):.4f}")